In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
from torchviz import make_dot
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.datasets as datasets

In [ ]:
%load_ext autoreload
%autoreload 2

from modules.Annealed_Importance_Sampling import 
from modules.Contrastive_Divergence_Minimisation import 

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [3]:
# define transform and prepare CIFAR10 data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=False, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

# check the data type
images, labels = next(iter(train_loader))
print("images:", type(images), " / shape:", images.shape)
print("labels:", type(labels), " / shape:", labels.shape)

images: <class 'torch.Tensor'>  / shape: torch.Size([64, 3, 32, 32])
labels: <class 'torch.Tensor'>  / shape: torch.Size([64])


In [4]:
# Define a CNN for analysis with 2048 -> 128 -> 32 -> 5 -> 5 -> 10
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super(CIFAR10CNN, self).__init__()
        
        # 1. convolutional layer
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.flatten = nn.Flatten()
        
        # 2. fully-connected layers
        self.fc1 = nn.Linear(2048, 128)
        self.fc2 = nn.Linear(128, 32)
        self.fc3 = nn.Linear(32, 5)
        self.fc4 = nn.Linear(5, 5)
        self.fc5 = nn.Linear(5, 10)

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = torch.tanh(self.fc3(x))
        x = torch.tanh(self.fc4(x))
        x = self.fc5(x)
        return x
        
    def get_activations(self, x):
        conv_out = self.features(x)
        flat = self.flatten(conv_out)
        
        act1 = torch.tanh(self.fc1(flat))
        act2 = torch.tanh(self.fc2(act1))
        act3 = torch.tanh(self.fc3(act2))
        act4 = torch.tanh(self.fc4(act3))
        act5 = self.fc5(act4)

        _, predicted = torch.max(act5, 1)

        return {
            'ConvOut': flat.detach().cpu().numpy(), # Convoluted features
            'L1': act1.detach().cpu().numpy(),
            'L2': act2.detach().cpu().numpy(),
            'L3': act3.detach().cpu().numpy(),
            'L4': act4.detach().cpu().numpy(),
            'L5': act5.detach().cpu().numpy(),
            'predicted': predicted.detach().cpu().numpy()
        }

In [8]:
# initialise model and optimizer
model = CIFAR10CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [9]:
epochs = 100

for epoch in range(epochs):
    # --- 1. learning phase---
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

    # --- 2. obtain activation ---
    conv_out_list, l1, l2, l3, l4, l5 = [], [], [], [], [], []
    predictions = []
    
    model.eval() 
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            
            # get activations
            activations = model.get_activations(images)
            conv_out_list.append(activations['ConvOut'])
            l1.append(activations['L1'])
            l2.append(activations['L2'])
            l3.append(activations['L3'])
            l4.append(activations['L4'])
            l5.append(activations['L5'])
            predictions.append(activations['predicted']) # result
            
    # list the result
    conv_out_data = np.vstack(conv_out_list)
    l1 = np.vstack(l1)
    l2 = np.vstack(l2)
    l3 = np.vstack(l3)
    l4 = np.vstack(l4)
    l5 = np.vstack(l5)
    predictions = np.concatenate(predictions)

Epoch [1/100], Loss: 1.9488
Epoch [2/100], Loss: 1.6400
Epoch [3/100], Loss: 1.3987
Epoch [4/100], Loss: 1.1563
Epoch [5/100], Loss: 0.9653
Epoch [6/100], Loss: 0.8330
Epoch [7/100], Loss: 0.7182
Epoch [8/100], Loss: 0.6286
Epoch [9/100], Loss: 0.5417
Epoch [10/100], Loss: 0.4756
Epoch [11/100], Loss: 0.4121
Epoch [12/100], Loss: 0.3610
Epoch [13/100], Loss: 0.3258
Epoch [14/100], Loss: 0.2880
Epoch [15/100], Loss: 0.2611
Epoch [16/100], Loss: 0.2326
Epoch [17/100], Loss: 0.2222
Epoch [18/100], Loss: 0.2114
Epoch [19/100], Loss: 0.1912
Epoch [20/100], Loss: 0.1849
Epoch [21/100], Loss: 0.1705
Epoch [22/100], Loss: 0.1727
Epoch [23/100], Loss: 0.1661
Epoch [24/100], Loss: 0.1488
Epoch [25/100], Loss: 0.1533
Epoch [26/100], Loss: 0.1393
Epoch [27/100], Loss: 0.1422
Epoch [28/100], Loss: 0.1461
Epoch [29/100], Loss: 0.1383
Epoch [30/100], Loss: 0.1302
Epoch [31/100], Loss: 0.1195
Epoch [32/100], Loss: 0.1318
Epoch [33/100], Loss: 0.1168
Epoch [34/100], Loss: 0.1283
Epoch [35/100], Loss: 0

In [10]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\nAccuracy on the test data: {100 * correct / total:.2f}%")


Accuracy on the test data: 71.04%
